# ATML PA4 GPU smoke

Short checks only. A failure here is cheaper than a full run.
Attention at length 256, two training updates of the gated model, the Task 3 split-prefill check, one KV prompt, and one WikiText window.

In [ ]:
import gzip
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path('/kaggle/working/ATML-PA4')
root = Path('/kaggle/input')
print('inputs', [path.name for path in root.iterdir()], flush=True)
for path in list(root.rglob('*'))[:40]:
    print('found', path, flush=True)
locked = next(root.rglob('c4_calibration_locked.jsonl*'))
source = locked.parents[2]
data = Path('/kaggle/working/pa4-data')
if data.exists():
    shutil.rmtree(data)
gzip_dirs = {'public_locked', 'kv_retrieval', 'mqar'}
for path in source.rglob('*'):
    if not path.is_file():
        continue
    rel = path.relative_to(source)
    target = data / rel
    if path.suffix == '.jsonl' and path.parent.name in gzip_dirs:
        target = data / rel.with_suffix('.jsonl.gz')
        target.parent.mkdir(parents=True, exist_ok=True)
        with path.open('rb') as raw, gzip.open(target, 'wb') as packed:
            shutil.copyfileobj(raw, packed)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
print('data root', data, flush=True)
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(['git', 'clone', 'https://github.com/ttqureshi/ATML-PA4.git', str(REPO)])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'huggingface_hub==0.36.2',
    'accelerate', 'peft', 'sentencepiece', 'protobuf',
])
env = os.environ.copy()
env['PA4_DATA_REPO'] = str(data)
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

steps = [
    ('task2', ['task2/run.py', 'attention', '--max-length', '256']),
    ('task2', ['task2/run.py', 'train', '--updates', '2', '--only', 'gated_K16_mixed', '--skip-test']),
    ('task3', ['task3/run.py', 'smoke']),
    ('task2', ['task2/run.py', 'kv', '--limit', '1', '--skip-pack-benchmark']),
    ('task1', ['task1/run.py', 'gpu-smoke', '--output', 'task1/runs/gpu-smoke']),
]
failures = []
for branch, args in steps:
    print('\n====', branch, ' '.join(args), '====', flush=True)
    subprocess.check_call(['git', '-C', str(REPO), 'checkout', branch])
    completed = subprocess.run([sys.executable, *args], cwd=REPO, env=env)
    print('exit', completed.returncode, flush=True)
    if completed.returncode != 0:
        failures.append((branch, args, completed.returncode))

out = Path('/kaggle/working/smoke_out')
out.mkdir(exist_ok=True)
for path in REPO.rglob('*.json'):
    if 'runs' in path.parts or 'results' in path.parts:
        dest = out / path.name
        shutil.copy(path, dest)
        print('saved', dest.name, dest.stat().st_size, flush=True)
if failures:
    print('SMOKE FAILURES', failures, flush=True)
    raise SystemExit(1)
print('SMOKE OK', flush=True)
